# Automated Email Sender

This notebook automatically sends an email from `kitademidun@gmail.com` based on subject and body specified in a JSON configuration.

## Import Required Libraries

Import the necessary libraries including `smtplib`, `email.mime` modules, and `json` for reading configuration.

In [1]:
import smtplib
import json
import os
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

## Load Support Tickets from JSON File

Load the Wealthsimple support ticket emails from the JSON file.

In [ ]:
# Load support tickets from JSON file
with open('support_tickets.json', 'r') as f:
    support_tickets = json.load(f)

print(f"Loaded {len(support_tickets)} support tickets")
print("\nSample ticket:")
print(json.dumps(support_tickets[0], indent=2))

Email configuration loaded:
{
  "subject": "Test Email from Jupyter Notebook",
  "body": "This is an automated email sent from a Jupyter notebook. Testing email functionality."
}


## Configure Email Settings

Set up email credentials and SMTP server settings for Gmail.

**Note**: You'll need to use an App Password for Gmail accounts with 2FA enabled. 
Create one at: https://myaccount.google.com/apppasswords

In [3]:
# Email configuration - loaded from .env file
SENDER_EMAIL = os.getenv("SENDER_EMAIL")
RECEIVER_EMAIL = os.getenv("RECEIVER_EMAIL")
PASSWORD = os.getenv("EMAIL_PASSWORD")

# SMTP settings for Hotmail/Outlook
SMTP_SERVER = os.getenv("SMTP_SERVER")
SMTP_PORT = int(os.getenv("SMTP_PORT"))

print(f"Email configured: {SENDER_EMAIL} → {RECEIVER_EMAIL}")

Email configured: kitademidun@gmail.com → kademidun@hotmail.com


## Send All Support Tickets via Email

Loop through all support tickets and send each one as an email.

In [ ]:
import time

# Loop through all support tickets and send emails
success_count = 0
failed_count = 0

for index, ticket in enumerate(support_tickets, 1):
    try:
        # Create the email message
        message = MIMEMultipart()
        message["From"] = SENDER_EMAIL
        message["To"] = RECEIVER_EMAIL
        message["Subject"] = ticket["subject"]
        
        # Attach the email body
        message.attach(MIMEText(ticket["body"], "plain"))
        
        # Connect to the SMTP server
        print(f"\n[{index}/{len(support_tickets)}] Sending: {ticket['subject']}")
        server = smtplib.SMTP(SMTP_SERVER, SMTP_PORT)
        server.starttls()
        
        # Login and send
        server.login(SENDER_EMAIL, PASSWORD)
        text = message.as_string()
        server.sendmail(SENDER_EMAIL, RECEIVER_EMAIL, text)
        server.quit()
        
        print(f"✅ Email sent successfully!")
        success_count += 1
        
        # Small delay between emails to avoid rate limiting
        if index < len(support_tickets):
            time.sleep(2)
        
    except smtplib.SMTPAuthenticationError:
        print(f"❌ Authentication failed for ticket {index}")
        failed_count += 1
    except smtplib.SMTPException as e:
        print(f"❌ SMTP error for ticket {index}: {str(e)}")
        failed_count += 1
    except Exception as e:
        print(f"❌ Error for ticket {index}: {str(e)}")
        failed_count += 1

print(f"\n{'='*50}")
print(f"Summary: {success_count} emails sent successfully, {failed_count} failed")
print(f"{'='*50}")

Connecting to SMTP server...
Logging in...
Logging in...
Sending email...
Sending email...
✅ Email sent successfully!
✅ Email sent successfully!
